# 9-Patch Coordinate Regression Training

Thin Colab notebook — logic lives in .py files.

In [ ]:
# Install deps & mount drive
!pip install -q timm albumentations

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Point at our training code (adjust path to your drive)
import sys
sys.path.insert(0, '/content/drive/MyDrive/deepslice/training')

In [ ]:
# Train
!python /content/drive/MyDrive/deepslice/training/train.py \
    --checkpoint-dir /content/drive/MyDrive/9patch_checkpoints_v2 \
    --data-dir /content/9patch_data_v2 \
    --batch-size 32

## Training Curves

In [ ]:
import torch
from visualize import plot_training_history

checkpoint = torch.load('/content/drive/MyDrive/9patch_checkpoints_v2/best_model.pth',
                        map_location='cpu', weights_only=False)
plot_training_history(checkpoint['history'])

## Visualize Predictions

In [ ]:
from datagen import NinePatchGeneratorV2
from model import NinePatchRegressor
from visualize import visualize_predictions

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NinePatchRegressor('efficientnet_b0', pretrained=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

generator = NinePatchGeneratorV2(output_size=(224, 224))
visualize_predictions(model, generator, device, n_samples=6, stage=4)

## Export

In [ ]:
# Export to TorchScript
model.eval()
model.cpu()

example_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, example_input)

export_path = '/content/drive/MyDrive/9patch_checkpoints_v2/ninepatch_model_v2.pt'
traced_model.save(export_path)
print(f'Exported TorchScript model to {export_path}')

# Verify
loaded = torch.jit.load(export_path)
test_output = loaded(example_input)
print(f'Verification - output shape: {test_output.shape}')